# Sentiment Analysis Service — Phase 2 finetuning

Notebook chạy các script training trong repo (tương thích pipeline DVC).

**MLflow / DagsHub (tùy chọn):** Các ô bên dưới cho phép gửi metric training lên DagsHub. Nếu bỏ qua, run vẫn chạy; MLflow chỉ ghi cục bộ (`mlruns`) trên Colab.


In [ ]:
!git clone https://github.com/RocketUp-Team/Sentiment-Analysis-Service.git
%cd Sentiment-Analysis-Service
!git checkout feature/absa-sarcasm-phase2
!pip install -r requirements.txt
!pip install datasets peft accelerate

In [ ]:
print("Downloading datasets...")
!python -m src.data.downloader --task sarcasm
!python -m src.data.downloader --task sentiment

### MLflow → DagsHub (tùy chọn)

`run_finetuning` đọc `MLFLOW_TRACKING_URI` và (với DagsHub) username + token theo chuẩn MLflow (`MLFLOW_TRACKING_USERNAME` / `MLFLOW_TRACKING_PASSWORD`).

**Trên Colab:** mở **Secrets** (biểu tượng khóa trên sidebar) và thêm:

| Secret | Giá trị |
|--------|--------|
| `MLFLOW_TRACKING_URI` | `https://dagshub.com/<USER>/<REPO>.mlflow` |
| `DAGSHUB_USER` | Tên đăng nhập DagsHub |
| `DAGSHUB_TOKEN` | Access token (không dùng mật khẩu web) |

Chạy ô code ngay dưới **trước** các ô training. Nếu không cấu hình, metric vẫn được log vào `file:./mlruns` trong thư mục repo.

Experiments trên DagsHub: `phase2_finetuning_sarcasm`, `phase2_finetuning_sentiment`.


In [ ]:
import os


def _colab_secret(key: str) -> str | None:
    try:
        from google.colab import userdata

        value = userdata.get(key)
        return value.strip() if value else None
    except Exception:
        return None


_uri = _colab_secret("MLFLOW_TRACKING_URI") or os.environ.get(
    "MLFLOW_TRACKING_URI", ""
).strip()
_user = (
    _colab_secret("DAGSHUB_USER")
    or os.environ.get("MLFLOW_TRACKING_USERNAME", "").strip()
    or os.environ.get("DAGSHUB_USER", "").strip()
)
_token = (
    _colab_secret("DAGSHUB_TOKEN")
    or os.environ.get("MLFLOW_TRACKING_PASSWORD", "").strip()
    or os.environ.get("DAGSHUB_TOKEN", "").strip()
)

if _uri:
    os.environ["MLFLOW_TRACKING_URI"] = _uri
    print("MLFLOW_TRACKING_URI set -> remote MLflow (e.g. DagsHub).")
else:
    print("MLFLOW_TRACKING_URI unset -> training logs to file:./mlruns in repo.")

if _uri and _user and _token:
    os.environ["MLFLOW_TRACKING_USERNAME"] = _user
    os.environ["MLFLOW_TRACKING_PASSWORD"] = _token
    print("MLFLOW_TRACKING_USERNAME / MLFLOW_TRACKING_PASSWORD set for DagsHub.")
elif _uri and (not _user or not _token):
    print(
        "Warning: URI set but DAGSHUB_USER or DAGSHUB_TOKEN missing -> "
        "DagsHub often returns 403. Add Colab Secrets or env vars."
    )

In [ ]:
print("Training Sarcasm Adapter...")
!python -m src.scripts.run_finetuning --task sarcasm

In [ ]:
print("Training Sentiment Adapter...")
!python -m src.scripts.run_finetuning --task sentiment

In [ ]:
print("Compressing models for download...")
!tar -czvf adapters.tar.gz models/adapters/